# Diagnóstico de cancro da mama — Active Learning com Human-in-the-Loop

**Inteligência Artificial Centrada no Humano** · Mestrado em Inteligência Artificial e Engenharia de Dados · FCTUC

Começa com poucas imagens anotadas e, em cada iteração, o modelo escolhe a imagem mais útil e pede a label ao humano. Compara três estratégias de query do modAL (`uncertainty`, `entropy`, `margin`) com vários batch sizes. Correr primeiro `01_passive_learning.ipynb` para ter a baseline.

> **Estado**
> - ✅ Feito: ciclo de Active Learning com as 3 estratégias, humano real ou simulado (o modelo aprende com a label dada pelo humano), comparação com a baseline; testado com uma configuração reduzida.
> - ⬜ Falta: correr as experiências completas (ainda não há resultados), Grad-CAM (#4) e conclusões.

**Índice**
1. Configuração
2. Dados
3. Experiências
4. Resultados
5. Conclusões

## O que é Active Learning?

Anotar ecografias exige um médico e é caro e demorado. No **Passive Learning** (notebook 01), o modelo recebe todas as imagens já anotadas. No **Active Learning**, começa com poucas imagens anotadas e é o próprio modelo que escolhe quais quer ver anotadas a seguir: as que lhe trazem mais informação, normalmente aquelas em que está mais indeciso. Assim, o objetivo é chegar a um desempenho parecido com o do Passive Learning com muito menos anotações.

### O ciclo, passo a passo
1. Treinar a CNN com um pequeno conjunto inicial de imagens anotadas (`N_INITIAL`).
2. As restantes imagens de treino formam a **pool** (imagens "sem label").
3. Em cada iteração:
   - o modelo calcula as probabilidades para todas as imagens da pool e, com uma **estratégia de query**, escolhe a mais útil;
   - o **humano** dá a classe correta dessa imagem (*Human-in-the-Loop*);
   - a imagem passa da pool para o conjunto anotado e o modelo volta a treinar;
   - registam-se a accuracy e a loss, no conjunto anotado e no teste.
4. Repetir durante `N_QUERIES` iterações.

### As três estratégias de query (biblioteca [modAL](https://modal-python.readthedocs.io/))
O modelo dá 3 probabilidades por imagem, por exemplo `[0.40, 0.35, 0.25]`. As estratégias medem a "indecisão" de formas diferentes:

| Estratégia | Escolhe a imagem com… | No exemplo |
|---|---|---|
| `uncertainty` | maior incerteza = 1 − probabilidade da classe mais provável | 1 − 0.40 = 0.60 |
| `margin` | menor diferença entre as duas classes mais prováveis | 0.40 − 0.35 = 0.05 |
| `entropy` | maior entropia (as probabilidades mais espalhadas pelas 3 classes) | −Σ p·log p |

### O papel do humano
Com `HUMAN = True`, cada imagem escolhida aparece no notebook com a sugestão do modelo, e a pessoa escreve a classe correta. **O modelo aprende com a resposta da pessoa**, mesmo que esteja errada. Com `HUMAN = False`, a resposta é simulada com a label verdadeira do dataset, para se poder correr as experiências sem ninguém ao teclado.

## 1. Configuração

A primeira célula prepara o ambiente e funciona tanto no **Google Colab** como **localmente**:
- no Colab, monta o Google Drive (onde deve estar uma cópia do repositório com o dataset) e instala as bibliotecas que faltam (`modAL-python`, `scikeras`);
- localmente, assume que o notebook foi aberto a partir da pasta `notebooks/`.

Depois junta a raiz do repositório ao `sys.path`, para se poder importar o código partilhado da pasta `src/`. Os caminhos do dataset e da pasta de resultados estão em `src/config.py`.

In [ ]:
# Ambiente: monta o Google Drive no Colab; localmente usa a pasta do repositório
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/My Drive/miacd-iach-breast-cancer-diagnosis")  # pasta do repositório no Drive
    %pip install -q modAL-python scikeras
else:
    ROOT = Path.cwd().parent

sys.path.insert(0, str(ROOT))
from src.config import DATA_PATH, RESULTS_PATH

### Parâmetros
- **`BATCH_SIZES`**: nº de imagens que a rede processa de cada vez **durante o treino** (o *batch size* do Keras), não o nº de imagens pedidas ao humano por iteração, que é sempre 1. Batches pequenos atualizam os pesos mais vezes, mas de forma mais ruidosa.
- **`N_INITIAL`**: imagens anotadas no início, escolhidas ao acaso.
- **`N_QUERIES`**: nº de iterações, ou seja, nº de imagens pedidas ao humano.
- **`EPOCHS`**: épocas de treino de cada vez que o modelo é treinado (no início e após cada nova label).

In [ ]:
# Parâmetros
IMG_SIZE = 224          # reduzir (ex.: 128) se faltar memória
STRATEGIES = ["uncertainty", "entropy", "margin"]
BATCH_SIZES = [16, 32, 64, 128]
N_INITIAL = 1000        # nº de imagens anotadas no início
N_QUERIES = 10          # nº de iterações de Active Learning (imagens pedidas ao humano)
EPOCHS = 10
HUMAN = False           # True: pede a label a uma pessoa; False: simula o humano com a label do dataset

## 2. Dados

`load_dataset` (em `src/data.py`) faz cinco passos:

1. **Ler** as imagens em escala de cinzento e redimensionar para `IMG_SIZE × IMG_SIZE`. Os valores dos píxeis passam de 0–255 para 0–1 (normalização), o que ajuda o treino a convergir.
2. **Tirar a classe do nome do ficheiro** (ex.: `benign (12).png` → `benign`). As labels numéricas são `0 = normal`, `1 = benign`, `2 = malignant`. As imagens duplicadas são excluídas: o dataset tem a mesma imagem como `benign (433)` e `malignant (145)`, e não se sabe qual das labels está certa. Ficam 778 imagens.
3. **Dividir em três conjuntos**, de forma estratificada (cada um fica com as mesmas proporções de classes):
   - **treino (65%)**: as imagens com que o modelo aprende;
   - **validação (15%)**: para acompanhar o treino e escolher parâmetros (épocas, batch size…);
   - **teste (20%)**: só para a avaliação final.
4. **Balancear só o treino** com rotações de 90º e reflexões, até cada classe ter um número parecido de imagens. A validação e o teste ficam com as imagens originais, para medir o desempenho em condições reais.
5. **Baralhar o treino**, para as classes ficarem misturadas.

> **Porque é que a divisão vem antes do balanceamento?** Se fosse ao contrário, uma imagem podia ir para o treino e a sua versão rodada para a validação ou o teste. O modelo seria avaliado com imagens que, na prática, já viu — isto chama-se *data leakage* e dá resultados otimistas e falsos. Assim, **nenhuma imagem de validação ou de teste, nem uma cópia dela, é usada no treino.**

In [ ]:
# Carregar, dividir em treino/validação/teste e balancear só o treino
import numpy as np
from src.data import CLASSES, load_dataset, summarize

x_train, x_val, x_test, y_train, y_val, y_test = load_dataset(DATA_PATH, size=IMG_SIZE)
summarize(x_train, x_val, x_test, y_train, y_val, y_test)

## 3. Experiências

Para cada combinação de estratégia e batch size, `run_active_learning` (em `src/active_learning.py`) corre o ciclo completo descrito no início, sempre com uma CNN nova, e devolve:
- `history`: accuracy e loss em cada iteração, no conjunto anotado (*train*) e no teste (*test*);
- `y_pred`: as previsões finais no conjunto de teste, usadas para calcular as métricas.

São 3 estratégias × 4 batch sizes = **12 experiências**, por isso esta célula pode demorar bastante. Os resultados ficam guardados em `results/active_learning.json`.

In [ ]:
from src.active_learning import run_active_learning
from src.evaluation import compute_metrics, save_json

results = {}   # results[estratégia][batch_size] = {"history", "y_pred", "metrics"}
for strategy in STRATEGIES:
    results[strategy] = {}
    for batch_size in BATCH_SIZES:
        print(f"{strategy} - batch {batch_size}")
        history, y_pred = run_active_learning(strategy, x_train, y_train, x_val, y_val, x_test, y_test,
                                              batch_size=batch_size, epochs=EPOCHS, n_initial=N_INITIAL,
                                              n_queries=N_QUERIES, human=HUMAN)
        results[strategy][batch_size] = {"history": history, "y_pred": y_pred.tolist(),
                                         "metrics": compute_metrics(y_test, y_pred)}

save_json(results, RESULTS_PATH / "active_learning.json")

## 4. Resultados

### 4.1 Evolução ao longo das iterações
Um gráfico por estratégia, com uma cor por batch size:
- **linha contínua**: accuracy; **linha tracejada fina**: loss;
- **à esquerda**: no conjunto anotado até ao momento (*train*); **à direita**: no conjunto de teste;
- **linha preta tracejada**: accuracy do Passive Learning (notebook 01), a referência a atingir.

Se a accuracy de teste subir ao longo das iterações, as imagens escolhidas estão a ajudar o modelo.

In [ ]:
from src.evaluation import load_json, plot_history

baseline_file = RESULTS_PATH / "passive_learning.json"
baseline = load_json(baseline_file) if baseline_file.exists() else None

for strategy in STRATEGIES:
    histories = {batch: r["history"] for batch, r in results[strategy].items()}
    plot_history(histories, f"Active Learning using {strategy}_sampling", baseline=baseline)

### 4.2 Métricas no conjunto de teste
As mesmas métricas do notebook 01 (ver a explicação lá), no fim das iterações. A primeira linha é a baseline de Passive Learning.

- `recall` e `F1`: média das 3 classes;
- `recall mal.`: recall da classe maligna;
- `FN`: falsos negativos (cancros classificados como benignos ou normais). **Quanto menor, melhor.**

In [ ]:
print(f"{'estratégia':<12}{'batch':>6}{'accuracy':>10}{'recall':>8}{'F1':>8}{'recall mal.':>13}{'FN':>5}")
rows = [("passive", "-", baseline)] if baseline is not None else []
rows += [(s, b, r["metrics"]) for s in STRATEGIES for b, r in results[s].items()]
for strategy, batch_size, m in rows:
    print(f"{strategy:<12}{batch_size:>6}{m['accuracy']:>10.3f}{m['recall_macro']:>8.3f}{m['f1_macro']:>8.3f}"
          f"{m['recall_malignant']:>13.3f}{m['false_negatives']:>5}")

### 4.3 Matrizes de confusão da estratégia com menos falsos negativos
Uma matriz por batch size. As linhas são a classe real e as colunas a prevista; os falsos negativos estão na linha `malignant`, nas colunas `normal` e `benign`.

In [ ]:
from src.evaluation import plot_confusion_matrix

best = min(STRATEGIES, key=lambda s: min(r["metrics"]["false_negatives"] for r in results[s].values()))
for batch_size, r in results[best].items():
    plot_confusion_matrix(y_test, r["y_pred"], title=f"{best}_sampling - batch {batch_size}")

## 5. Conclusões

*A preencher depois de correr as experiências completas.*